<a href="https://colab.research.google.com/github/KathituCodes/Legal-Q-A-Assistant-RAG-and-OpenAI/blob/main/Legal_QA_Assistant_RAG_OpenAI_debugged.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***Problem Statement***

<font color="green">**Legal professionals, businesses, and individuals often need to extract specific clauses or insights from lengthy legal contracts and documents**. Manually searching through hundreds of pages is **time-consuming, error-prone, and inefficient. Traditional keyword-based search tools fail to capture the semantic meaning of queries, making it difficult to find relevant information (e.g., “penalty clause,” “termination conditions,” “confidentiality agreements”).**

This project aims to **develop a Legal Document Q\&A Assistant** using **Retrieval-Augmented Generation (RAG)** and **OpenAI’s LLMs**. The system will allow users to upload legal documents (e.g., contracts, agreements, policies) and query them in natural language. By combining semantic retrieval (vector search) with generative AI, the assistant will provide **accurate, context-aware answers**, helping users quickly understand critical contract terms without needing to read entire documents.


# ***Experimental Design***

### **Objective**

To design and evaluate a Retrieval-Augmented Generation (RAG) system that enables accurate Q\&A over unstructured legal documents using OpenAI’s GPT models.

### **1. Data Sources**

* **CUAD Dataset** (Contract Understanding Atticus Dataset) — annotated legal clauses.
* Supplementary: Manually sourced open legal PDFs (contracts, NDAs, policies).


### **2. System Architecture**

**Pipeline Design:**

1. **Document Ingestion** → Upload PDF / load dataset
2. **Preprocessing** → Chunk documents into passages (1000 tokens with overlap)
3. **Vectorization** → Encode text using OpenAI embeddings
4. **Storage** → Store embeddings in **FAISS (vector DB)**
5. **Retrieval** → Find top-k most relevant chunks per user query
6. **Generation** → Send retrieved context + query to GPT (via OpenAI API)
7. **Response** → Return structured, human-readable answer


### **3. Experimental Setup**

* **Tools:** Python, LangChain, OpenAI API, FAISS, Streamlit
* **Models:** GPT-3.5-turbo (baseline), GPT-4 (advanced comparison)
* **Chunking Strategy:** Compare 500 vs 1000 token chunks with overlap
* **Retrieval k-values:** Test top-3 vs top-5 vs top-10 documents retrieved
* **Evaluation Metrics:**

  * **Accuracy** — % of correct answers vs gold labels (using CUAD annotated answers).
  * **Relevance Score** — how well retrieved chunks match query intent.
  * **Response Coherence** — measured via human evaluation (clarity, usefulness).
  * **Latency** — average response time per query.


In [ ]:
# Install dependencies
# (required for hf:// parquet loading), langchain-openai and langchain-community
# (required for current LangChain import paths), and pyngrok (replaces unreliable localtunnel).
!pip install openai langchain langchain-openai langchain-community \
             faiss-cpu streamlit pypdf \
             huggingface_hub pyarrow pyngrok


Above I started by setting up the environment for the RAG system, ensuring tools for embeddings, retrieval, and UI are available. Critical for enabling document processing and query handling in the legal assistant.

# ***Preprocess Documents***

Use LangChain to load and split text into chunks.

- Link to dataset: https://huggingface.co/datasets/dvgodoy/CUAD_v1_Contract_Understanding_PDF?utm_source=chatgpt.com


In [ ]:
import pandas as pd

df = pd.read_parquet("hf://datasets/dvgodoy/CUAD_v1_Contract_Understanding_PDF/data/train-00000-of-00001.parquet")

# Verify available columns before accessing them. Different parquet versions
# may use different column names. This prevents a silent KeyError downstream.
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head(3)


I provided the core dataset for the legal assistant. The CUAD dataset’s annotations enable evaluation of clause-specific queries (e.g., penalty or termination clauses), aligning with the project’s goal of precise information extraction.

> **Note:** `langchain-community` is now installed in the first cell above alongside all other dependencies. The separate mid-notebook install cell has been removed to avoid redundancy and version conflicts.

The above ensures access to the latest LangChain features, such as advanced text splitting or document handling, which are vital for preprocessing legal documents in the RAG pipeline.

In [ ]:
# Imports text splitting and document classes from LangChain.
# Extracts text from the DataFrame's text column.
# Converts each text into a LangChain Document object.
# Splits documents into chunks of 1000 tokens with 200-token overlap using RecursiveCharacterTextSplitter.

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Guard against unexpected column names. The column verification in the
# cell above should confirm 'text' exists before this cell runs.
if 'text' not in df.columns:
    raise KeyError(
        f"Expected column 'text' not found. Available columns: {df.columns.tolist()}. "
        "Update the column name below to match your dataset."
    )

texts = df['text'].tolist()

# Convert text strings to Document objects
docs = [Document(page_content=t) for t in texts]

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(docs)

print(f"Total chunks created: {len(chunks)}")
print(f"Sample chunk (first 200 chars): {chunks[0].page_content[:200]}")


- **Chunking** means splitting a long document into smaller, manageable pieces so the system can search through them more effectively.
- **Overlap** means a little bit of text is repeated between chunks to keep the context flowing, which helps the system understand the full meaning.

This is important because it makes it easier for the system to correctly find and interpret **legal clauses** (like “confidentiality agreements”) during search.


# ***Create Vector Store***

Embed chunks into FAISS for similarity search.

In [ ]:
# Imports os for environment variable handling and userdata for Colab secrets.
# Sets the OpenAI API key as an environment variable.
# Retrieves the key into the api_key variable for use in subsequent API calls.

import os
from google.colab import userdata


#
# Store your key in Colab Secrets (key icon in the left
# sidebar), name it OPENAI_API_KEY, then load it securely as shown below.
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Verify the key was loaded (prints True/False without exposing the key value)
api_key = os.environ.get("OPENAI_API_KEY", "")
print("API key loaded:", bool(api_key) and api_key != '"PASTE API KEY HERE"')


I the proceeded to securely configure access to OpenAI’s embedding and LLM services, a prerequisite for vectorization and answer generation in the RAG pipeline.

In [ ]:
# Imports OpenAI embeddings and FAISS vector store.
# Initializes OpenAI embeddings for text vectorization.
# Creates a FAISS vector store from the chunked documents using the embeddings.

from langchain_openai import OpenAIEmbeddings           # was: langchain.embeddings
from langchain_community.vectorstores import FAISS      # was: langchain.vectorstores

import os

FAISS_INDEX_PATH = "faiss_legal_index"

embeddings = OpenAIEmbeddings()

# Embedding all 50,000+ chunks in one shot costs $1-3 and takes 10-30 mins
# with no feedback. This block saves the index to disk after the first build so
# subsequent runs load instantly without re-calling the API.
if os.path.exists(FAISS_INDEX_PATH):
    print("Loading existing FAISS index from disk...")
    db = FAISS.load_local(
        FAISS_INDEX_PATH,
        embeddings,
        allow_dangerous_deserialization=True
    )
    print("Index loaded successfully.")
else:
    print(f"Building FAISS index from {len(chunks)} chunks. This calls the OpenAI")
    print("embeddings API and may take 10-30 minutes on the full dataset.")
    print("The index will be saved to disk so this only runs once.")
    db = FAISS.from_documents(chunks, embeddings)
    db.save_local(FAISS_INDEX_PATH)
    print(f"Index built and saved to '{FAISS_INDEX_PATH}'.")


If you get an error that indicates a need for API quota management. When successful, the vector store enables fast retrieval of relevant document chunks, critical for answering queries like “termination conditions.”

# ***Build Retrieval-Augmented Generation (RAG)***

Pipeline: user query → retrieve docs → OpenAI LLM response.

In [ ]:
# Imports RetrievalQA chain and ChatOpenAI model.
# Converts the FAISS vector store into a retriever.
# Initializes GPT-3.5-turbo as the LLM.
# Builds a RetrievalQA chain to combine retrieval and generation.
# Runs a sample query about penalty clauses and prints the response.

from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI   # was: langchain.chat_models

retriever = db.as_retriever(search_kwargs={"k": 5})
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    return_source_documents=True
)

query = "What is the penalty clause in this contract?"

# qa_chain.run() is deprecated and removed in langchain>=0.2.
# Correct call is .invoke() which returns a dict with 'result' and 'source_documents'.
result = qa_chain.invoke({"query": query})
print("Answer:")
print(result["result"])
print("\nSource chunks used:")
for i, doc in enumerate(result["source_documents"], 1):
    print(f"  [{i}] {doc.page_content[:150]}...")


The RAG pipeline works in **two main steps**:

1. **Retrieve** – It first searches and pulls out the most relevant pieces (“chunks”) of text from a large collection of documents.
2. **Generate** – Then it uses those chunks to create a clear and precise answer.

This is especially useful because it means the system can quickly find and explain the exact **legal clauses** you need, instead of making you read through long contracts yourself.


# ***Create Streamlit App***

In [ ]:
%%writefile app.py
# Streamlit runs app.py as a separate process so notebook variables do not
# exist there.
# The entire RAG setup is now defined inside app.py. @st.cache_resource ensures
# the chain is only built once per Streamlit session, not on every interaction.

import os
import streamlit as st
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import pandas as pd

# Load API key from environment (set before launching Streamlit)
# In Colab this is set via: os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    st.error("OPENAI_API_KEY environment variable is not set. Set it before launching this app.")
    st.stop()

FAISS_INDEX_PATH = "faiss_legal_index"

@st.cache_resource
def build_chain():
    embeddings = OpenAIEmbeddings()
    if os.path.exists(FAISS_INDEX_PATH):
        db = FAISS.load_local(
            FAISS_INDEX_PATH,
            embeddings,
            allow_dangerous_deserialization=True
        )
    else:
        df = pd.read_parquet(
            "hf://datasets/dvgodoy/CUAD_v1_Contract_Understanding_PDF/"
            "data/train-00000-of-00001.parquet"
        )
        texts = df['text'].tolist()
        docs = [Document(page_content=t) for t in texts]
        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        chunks = splitter.split_documents(docs)
        db = FAISS.from_documents(chunks, embeddings)
        db.save_local(FAISS_INDEX_PATH)
    retriever = db.as_retriever(search_kwargs={"k": 5})
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
    return RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
        chain_type="stuff",
        return_source_documents=True
    )

st.set_page_config(page_title="Legal Q&A Assistant", page_icon="⚖️")
st.title("⚖️ Legal Q&A Assistant")
st.markdown("Ask any question about the legal contracts in the CUAD dataset.")

query = st.text_input("Your question:", placeholder="e.g. What is the penalty clause?")

if query:
    with st.spinner("Searching documents and generating answer..."):
        qa_chain = build_chain()
        # FIX: .run() is removed in current LangChain. Use .invoke() instead.
        result = qa_chain.invoke({"query": query})
    st.write("### Answer")
    st.write(result["result"])
    with st.expander("Source passages used"):
        for i, doc in enumerate(result["source_documents"], 1):
            st.markdown(f"**Chunk {i}:** {doc.page_content[:300]}...")


Streamlit will enables interactive Q&A.


- The system gives a **user-friendly interface** that makes it easy to search and ask questions about legal documents.
- This supports the projects goal of making the tool **accessible to non-technical users**—like lawyers—who may not have technical skills but still need quick answers.


> **Note:** The `icanhazip.com` IP fetch cell has been removed. It was only needed to supply the localtunnel password, which has been replaced by pyngrok in the cell below.

In [ ]:
# pyngrok gives a stable public HTTPS URL directly.
#
# Before running: make sure OPENAI_API_KEY is set in the environment
# (it was set in the API key cell above and persists for this Colab session).

from pyngrok import ngrok
import threading, time

# Start Streamlit in the background
def run_streamlit():
    import subprocess
    subprocess.run(["streamlit", "run", "app.py", "--server.port", "8501",
                    "--server.headless", "true"])

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()
time.sleep(4)  # Give Streamlit a moment to start

# Open a public tunnel to port 8501
public_url = ngrok.connect(8501)
print("Legal Q&A Assistant is live at:", public_url)
